# 02 — Baseline & Model Comparison

Цель: обучить baseline (Logistic Regression), сравнить несколько моделей и построить ансамбль.

Весь код моделирования вынесен в `src/modeling.py`; здесь мы вызываем его функции и анализируем результаты.

**Метрики:** F1-score (приоритет) и ROC-AUC.

## 0. Настройка окружения

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Корень проекта: {project_root}')

In [ ]:
from src.modeling import (
    load_splits,
    train_baseline,
    train_all_models,
    train_ensemble,
    evaluate,
    save_model,
)

## 1. Загрузка данных

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_splits(project_root)

## 2. Baseline: Logistic Regression

Обучаем на исходных признаках (без engineered), без подбора гиперпараметров.

In [ ]:
baseline_model, baseline_metrics = train_baseline(X_train, y_train, X_val, y_val)
print(f"\nBaseline F1 (val): {baseline_metrics['f1']:.4f}")
print(f"Baseline ROC-AUC (val): {baseline_metrics['roc_auc']:.4f}")

## 3. Сравнение моделей

Обучаем RandomForest, XGBoost, LightGBM, GradientBoosting на полном наборе признаков (включая engineered).

In [ ]:
pipelines, results_df = train_all_models(X_train, y_train, X_val, y_val)
display(results_df)

## 4. Ансамбль (Soft Voting)

Объединяем все обученные модели с помощью Soft Voting.

In [ ]:
ensemble, ensemble_metrics = train_ensemble(pipelines, X_train, y_train, X_val, y_val)
print(f"\nEnsemble F1 (val): {ensemble_metrics['f1']:.4f}")
print(f"Ensemble ROC-AUC (val): {ensemble_metrics['roc_auc']:.4f}")

## 5. Выбор лучшей модели и тестовая оценка

In [ ]:
import pandas as pd

# Сводная таблица всех моделей
summary = results_df.copy()
summary.loc['Baseline (LR)'] = baseline_metrics
summary.loc['Ensemble'] = ensemble_metrics
summary = summary.sort_values('f1', ascending=False)

print('Сводная таблица (Val):')
display(summary)

# Выбираем лучшую модель по F1
best_name = summary['f1'].idxmax()
print(f'\nЛучшая модель по F1: {best_name}')

if best_name == 'Ensemble':
    best_model = ensemble
elif best_name == 'Baseline (LR)':
    best_model = baseline_model
else:
    best_model = pipelines[best_name]

In [ ]:
# Финальная оценка на тестовой выборке
print('--- Финальная оценка на TEST ---')
test_metrics = evaluate(best_model, X_test, y_test, label=f'Best: {best_name}')
print(f"Test F1: {test_metrics['f1']:.4f} | Test ROC-AUC: {test_metrics['roc_auc']:.4f}")

In [ ]:
# Сохраняем лучшую модель
safe_name = best_name.replace(' ', '_').replace('(', '').replace(')', '')
save_model(best_model, safe_name, project_root)